In [ ]:
!date

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

### first

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
mcg = sc.read_h5ad(f'{indir}/h5ad/chrom100k/merged_less_obs.h5ad')

In [ ]:
mcg_df = pd.DataFrame(index=mcg.obs.index)
mcg_df['donor'] = mcg.obs['line']
mcg_df['umap0'] = mcg.obsm['X_umap'][:,0]
mcg_df['umap1'] = mcg.obsm['X_umap'][:,1]
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures'
mcg_df.to_csv(f'{outdir}/csv/figure2/mcg_donor_umap.csv', sep='\t')

In [ ]:
leiden_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/integrate_m3c/leiden_clusters_renamed.csv'
leiden = pd.read_csv(leiden_path, sep='\t', header=0, index_col=0)
leiden['renamed_leiden'] = [x.replace('Fibro', 'Early') for x in leiden['renamed_leiden']]
leiden['renamed_leiden'] = leiden['renamed_leiden'].replace({'Inter_2' : 'Early_15'})
leiden['donor_leiden'] = [f'{x}_{y}' for x,y in zip(leiden['line'], leiden['renamed_leiden'])]
leiden.shape

In [ ]:
adata_list = []
for d in donors:    
    tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/chrom100k/{d}.h5ad')
    tmp_adata.obs['time'] = tmp_adata.obs['time'].astype(str)
    tmp_adata.obs['mc_cluster'] = leiden.reindex(tmp_adata.obs.index)['renamed_leiden']
    adata_list.append(tmp_adata)

In [ ]:
for adata in adata_list:
    sc.pl.umap(adata, color='time', groups='0')

In [ ]:
for adata in adata_list:
    sc.pl.umap(adata, color='mc_cluster')

In [ ]:
df_list = []

for adata in adata_list:
    
    tmp_df = pd.DataFrame(index=adata.obs.index)
    tmp_df['umap0'] = adata.obsm['X_umap'][:,0]
    tmp_df['umap1'] = adata.obsm['X_umap'][:,1]

    select = ['time', 'line', 'mc_cluster']
    for s in select:
        tmp_df[s] = adata.obs[s]
    df_list.append(tmp_df)
    
umap_df = pd.concat(df_list)
umap_df.rename(columns={'line' : 'donor'}, inplace=True)
umap_df.shape

In [ ]:
umap_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2'
umap_df.to_csv(f'{indir}/snmct_mcg_donor_day0_umap.csv', sep='\t')

In [ ]:
!date